# Chapter 8: Web Scraping
### Lab notebook for *Dive Into Data Science* (Bradford Tuckfield), pp. 169–189

**What we're doing today:** building datasets out of web pages. When nobody hands you a CSV, you can download a page's HTML with Python, pull out the pieces you care about, and load them into pandas.

**The plan:**

1. Download a page's source code with `requests`
2. Find things in it the naive way (`find()` + slicing) and see why that breaks
3. Learn **regular expressions** to search for *patterns* instead of exact text
4. Turn scraped results into a pandas DataFrame and a CSV
5. Use **Beautiful Soup** to pull out links, labeled fields, and tables by HTML structure
6. Talk through the advanced topics (dynamic pages, blocking, APIs, timing)

**How to use this notebook:** run the cells top to bottom. Some cells are *meant* to raise an error; each one is flagged with ⚠️ first. Because of those, **Run All will stop at the first flagged cell**, so run cell by cell (Shift+Enter).

All code is from the chapter. The only changes: regex patterns containing backslashes are written as raw strings (`r'...'`) so newer Python versions don't print escape-sequence warnings, and the searches that intentionally fail are split into their own cells.

## 0. Setup

Run this once if the packages aren't installed. Note the package is called `beautifulsoup4` but you import it as `bs4`. `lxml` is the parser Beautiful Soup uses under the hood.

In [ ]:
%pip install -q requests beautifulsoup4 lxml pandas

## 1. How websites work

Every web page is really a text file of **source code** (mostly HTML and JavaScript). Your browser reads that code and draws the page you see.

Try it: open https://nostarch.com, right-click on blank space, and choose **View Page Source**. That wall of text is what a scraper works with. We never see the pretty version; we only get the raw code.

## 2. Your first web scraper

The simplest possible scraper: give it a URL, get the page's code back, print the start of it.

- `requests.get(url)` downloads the page
- `.text` is the whole page's HTML as one long string
- `[0:600]` just keeps the first 600 characters so we don't flood the screen

In [ ]:
import requests
urltoget = 'https://bradfordtuckfield.com/indexarchive20210903.html'
pagecode = requests.get(urltoget)
print(pagecode.text[0:600])

**Reading the output.** HTML is made of **elements** wrapped in angle brackets. Look for `<title>` (a *start tag*) and `</title>` (an *end tag*): the text between them, `Bradford Tuckfield`, is what shows up on the browser tab.

We don't need to be HTML experts. We just need to recognize tags well enough to find the data sitting between them.

### ⚠️ Before scraping anything real

- **Read the site's terms of use.** Many sites forbid scraping, and some have sued over it.
- **Check for an API first.** Sites that are happy to share data often provide one.
- **Don't hammer the site.** Too many fast requests can crash it.

Everything in this lab scrapes the author's own demo pages, which he explicitly allows.

## 3. Parsing HTML the naive way

Downloading a page is easy. The real job is **parsing**: finding the one piece of data we want inside thousands of characters of code.

**Goal:** grab the email address from https://bradfordtuckfield.com/contactscrape.html (open it in a browser to see what it looks like).

### 3a. Search for the text `Email:`

`find()` returns the *position* of the text in the string.

In [ ]:
urltoget = 'https://bradfordtuckfield.com/contactscrape.html'
pagecode = requests.get(urltoget)

mail_beginning=pagecode.text.find('Email:')
print(mail_beginning)

`Email:` starts at character 511. Let's look at the 80 characters starting there:

In [ ]:
print(pagecode.text[(mail_beginning):(mail_beginning+80)])

There's a `<label>` tag sitting between `Email:` and the address. To get only the address we have to count characters by hand: skip 38, stop at 64.

In [ ]:
print(pagecode.text[(mail_beginning+38):(mail_beginning+64)])

It works, but **only on this exact page**. A different tag or a longer address and 38/64 are wrong.

### 3b. Search for `@` instead

Every email has an `@`, and there are no HTML tags inside an address.

In [ ]:
urltoget = 'https://bradfordtuckfield.com/contactscrape.html'
pagecode = requests.get(urltoget)

at_beginning=pagecode.text.find('@')
print(at_beginning)

In [ ]:
print(pagecode.text[(at_beginning-4):(at_beginning+22)])

Same problem: we had to know there are 4 characters before the `@` and 22 after. That can't be automated across many pages.

**What we really want** is to search for a *pattern*:

`<stuff that looks like the start of an email>` `@` `<stuff that looks like the end of an email>`

That's what regular expressions are for.

## 4. Regular expressions

A **regular expression** (regex) is a special string that describes a pattern to search for. Python's built-in `re` module handles them.

`re.search(pattern, text)` finds the first match, and `.span()` tells us where it starts and ends. The `r` before a string means *raw string*: Python passes it to `re` untouched.

In [ ]:
import re

print(re.search(r'recommend','irrelevant text I recommend irrelevant text').span())

`(18, 27)`: found, from index 18 to 27. So far this is no better than `find()`.

### The `+` metacharacter: one or more

A **metacharacter** has a special meaning instead of being matched literally. `+` means "one or more of the character before me". So `rec+om+end` matches *recommend* with any number (at least one) of c's and m's.

In [ ]:
import re
print(re.search('rec+om+end', 'irrelevant text I recommend irrelevant text').span())

Now try it on misspellings. These two match (`recomend` has one c and one m; `reccommend` has two of each):

In [ ]:
import re
print(re.search('rec+om+end','irrelevant text I recomend irrelevant text').span())
print(re.search('rec+om+end','irrelevant text I reccommend irrelevant text').span())

These two do **not** match. Before running: can you say why? (`reommend` has zero c's, and `+` needs at least one. `recomment` ends in t, and the pattern demands a d.)

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
print(re.search('rec+om+end','irrelevant text I reommend irrelevant text').span())

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
print(re.search('rec+om+end','irrelevant text I recomment irrelevant text').span())

**Takeaway:** a regex is exactly as flexible as you write it, no more.

### `*`: zero or more

Like `+`, but zero repetitions is fine too. `10*` means "a 1 followed by any number of 0s, including none".

In [ ]:
re.search('10*','My bank balance is 100').span()

In [ ]:
import re
print(re.search('10*','My bank balance is 1').span())
print(re.search('10*','My bank balance is 1000').span())
print(re.search('10*','My bank balance is 1000000').span())

`9000` has plenty of 0s but no `1` in front of them, so no match:

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
print(re.search('10*','My bank balance is 9000').span())

### `?`: zero or one

Use it when a character is optional. `Clarke?` matches *Clark* or *Clarke*.

In [ ]:
print(re.search('Clarke?','Please refer questions to Mr. Clark').span())

### Escape sequences: when you want the literal character

Metacharacters can bite. Say we want to find the equation `99+12=111`:

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
re.search('99+12=111','Example addition: 99+12=111').span()

It's right there in the text, so why no match? Because `+` was read as "one or more 9s", not as a plus sign. This *does* match:

In [ ]:
re.search('99+12=111','Incorrect fact: 999912=111').span()

To search for a real plus sign, put a backslash in front: `\\+`. The backslash is the **escape character**; it lets `+` "escape" its special meaning.

In [ ]:
re.search(r'99\+12=111','Example addition: 99+12=111').span()

Same trick for a literal question mark. This matches `Clarke?` but would not match `Clark?`:

In [ ]:
re.search(r'Clarke\?','Is anyone here named Clarke?').span()

And to search for a backslash itself, you need two:

In [ ]:
re.search(r'\\',r'The escape character is \\').span()

### Escape sequences that *add* meaning

A backslash in front of certain ordinary letters makes them special:

| Sequence | Matches |
|---|---|
| `\d` | any digit 0–9 |
| `\D` | anything that is **not** a digit |
| `\s` | whitespace (space, tab, newline) |
| `\w` | a "word" character: letter, digit, or underscore |

In [ ]:
re.search(r'\d','The loneliest number is 1').span()

### Square brackets: character classes

`[a-z]` means "any one lowercase letter". `[A-Z]` means "any one uppercase letter".

In [ ]:
re.search('[a-z]','My Twitter is @fake; my email is abc@def.com').span()

`(1, 2)` is the `y` in *My*, the first lowercase letter.

In [ ]:
re.search('[A-Z]','My Twitter is @fake; my email is abc@def.com').span()

`(0, 1)` is the `M`.

### Alternatives

Not sure how something is spelled? Allow either ending:

In [ ]:
re.search('Manchac[a|k]','Lets drive on Manchaca.').span()

*Side note:* inside square brackets you don't actually need the pipe; `[ak]` already means "a or k". (`[a|k]` works but would also match a literal `|`.) The pipe is the "or" operator *outside* brackets, e.g. `Manchaca|Manchack`.

### Combining metacharacters

| Symbol | Meaning |
|---|---|
| `$` | end of the line/string |
| `^` | beginning of the line/string |
| `.` | wildcard: any single character (except newline) |

**Scenario:** you're hunting for a PDF whose name starts with *school* but you forget the rest.

`school.*\.pdf$` reads as: `school`, then `.*` (any characters, any number of them), then `\.` (a real period), then `pdf`, then `$` (and nothing after).

In [ ]:
re.search(r'school.*\.pdf$','schoolforgottenname.pdf').span()

These three match:

In [ ]:
import re
print(re.search(r'school.*\.pdf$','schoolforgottenname.pdf').span())
print(re.search(r'school.*\.pdf$','school.pdf').span())
print(re.search(r'school.*\.pdf$','schoolothername.pdf').span())

These three don't. Predict why before you run each one: doesn't start with *school*; no period before *pdf*; doesn't *end* in `.pdf`.

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
print(re.search(r'school.*\.pdf$','othername.pdf').span())

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
print(re.search(r'school.*\.pdf$','schoolothernamepdf').span())

⚠️ **Expected error.** The next cell is *supposed* to fail with `AttributeError: 'NoneType' object has no attribute 'span'`. That is what "no match found" looks like: `re.search()` returned `None`, and `None` has no `.span()`.

In [ ]:
print(re.search(r'school.*\.pdf$','schoolforgottenname.pdf.exe').span())

More practice and reference: https://docs.python.org/3/howto/regex.html

### Back to email addresses

Now we can write the pattern we wanted in Section 3:

`[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+`

| Piece | Meaning |
|---|---|
| `[a-zA-Z]+` | one or more letters (the name) |
| `@` | a literal @ |
| `[a-zA-Z]+` | one or more letters (the domain, e.g. *gmail*) |
| `\.` | a literal period |
| `[a-zA-Z]+` | one or more letters (*com*, *org*, ...) |

In [ ]:
re.search(r'[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+',\
          'My Twitter is @fake; my email is abc@def.com').span()

Notice it skipped `@fake` (nothing before the @, no `.com` after) and found the real address.

**Limitation:** this pattern only allows letters, so it misses addresses with digits (`harry99@...`), dots, or non-English characters. A truly complete email regex is enormous. Good discussion question: how would you extend this one to allow digits?

## 5. Converting results to usable data

We're data scientists, not just scrapers, so the scraped values need to end up in a DataFrame.

**Goal:** collect *every* email address on https://bradfordtuckfield.com/contactscrape2.html

Step 1, download the page (same as before):

In [ ]:
import requests
urltoget = 'https://bradfordtuckfield.com/contactscrape2.html'
pagecode = requests.get(urltoget)

Step 2, find **all** matches. `re.search()` stops at the first match; `re.finditer()` gives us every one.

In [ ]:
allmatches=re.finditer(r'[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+',pagecode.text)

Step 3, collect the matches into a list. `match[0]` is the matched text itself.

In [ ]:
alladdresses = []
for match in allmatches:
    alladdresses.append(match[0])

print(alladdresses)

Step 4, turn the list into a pandas DataFrame:

In [ ]:
import pandas as pd
alladdpd=pd.DataFrame(alladdresses)
print(alladdpd)

Step 5, now it's ordinary pandas. Sort in reverse alphabetical order and export to CSV:

In [ ]:
alladdpd=alladdpd.sort_values(0,ascending=False)
alladdpd.to_csv('alladdpd20220720.csv')

**The full pipeline we just built:** URL → HTML → regex → list → DataFrame → CSV.

Regex can handle any scraping task in principle, but writing patterns for messy HTML gets painful. Next: a library that understands HTML structure for us.

## 6. Beautiful Soup

**Beautiful Soup** parses HTML so we can ask for elements by *tag name* and *attributes* instead of writing regex.

### 6a. Collecting every link on a page

Links in HTML are **anchor elements**:

```html
<a href='https://bradfordtuckfield.com'>Click here</a>
```

- `<a>` ... `</a>` marks the link
- `href` is an **attribute** holding the URL the link points to
- `Click here` is the text the visitor sees

`soup.find_all('a')` returns every anchor element; `each['href']` reads its URL.

*Heads up:* newer versions of Beautiful Soup may print an `XMLParsedAsHTMLWarning` here, because this particular page declares itself as XML at the top. It's only a warning; the links still print below it.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = 'https://bradfordtuckfield.com/indexarchive20210903.html'

response = requests.get(URL)
soup = BeautifulSoup(response.text, 'lxml')

all_urls = soup.find_all('a')
for each in all_urls:
    print(each['href'])

### 6b. Pulling out labeled fields

Back to the contact page from Section 3. Its HTML looks like this:

```html
<div class="find-widget">
    Email: <label class="email" href="#">demo@bradfordtuckfield.com</label>
</div>
```

The `class` attribute tells us what each `<label>` holds. So instead of counting characters, we just ask for "the label whose class is email". `find()` returns the first match; `.text` is the content between the tags.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = 'https://bradfordtuckfield.com/contactscrape.html'

response = requests.get(URL)
soup = BeautifulSoup(response.text, 'lxml')

email = soup.find('label',{'class':'email'}).text
mobile = soup.find('label',{'class':'mobile'}).text
website = soup.find('a',{'class':'website'}).text

print("Email : {}".format(email))
print("Mobile : {}".format(mobile))
print("Website : {}".format(website))

Compare this to Section 3: no magic numbers 38 and 64, and it keeps working if the address changes length.

### 6c. Scraping a table

Open https://bradfordtuckfield.com/user_detailsscrape.html to see the table. Its HTML:

```html
<table style="width:100%">
  <tr class="user-details-header">
    <th>Firstname</th> <th>Lastname</th> <th>Age</th>
  </tr>
  <tr class="user-details">
    <td>Jill</td> <td>Smith</td> <td>50</td>
  </tr>
  ...
</table>
```

- `<table>` = the whole table
- `<tr>` = **t**able **r**ow
- `<th>` = header cell
- `<td>` = **t**able **d**ata (one cell)

**Strategy:** find every row with class `user-details` (this skips the header row), then within each row find every `<td>`.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = 'https://bradfordtuckfield.com/user_detailsscrape.html'

response = requests.get(URL)
soup = BeautifulSoup(response.text, 'lxml')

all_user_entries = soup.find_all('tr',{'class':'user-details'})
for each_user in all_user_entries:
    user = each_user.find_all("td")
    print("User Firstname : {}, Lastname : {}, Age: {}"\
          .format(user[0].text, user[1].text, user[2].text))

### ✏️ Your turn

The chapter prints the table but suggests putting it in a DataFrame instead. Do that: build a DataFrame with columns `Firstname`, `Lastname`, `Age` from `all_user_entries`, then sort it by age.

*Hint:* follow the Section 5 recipe. Start with an empty list, append one `[first, last, age]` list per row, then `pd.DataFrame(rows, columns=[...])`. Ages come out as strings, so convert with `.astype(int)` before sorting.

In [ ]:
# Your code here

## 7. Advanced scraping (discussion)

No code here, just what to know exists:

- **Dynamic pages.** Many sites build their content with JavaScript after the page loads. `requests` + Beautiful Soup only see the static HTML, so the data may simply not be there. The **Selenium** library drives a real browser and can click, scroll, and fill in forms.
- **Getting blocked.** Sites that don't want to be scraped will block you. The safest response is to stop. Proxy servers and rotating IP addresses can get around blocks, but ask yourself whether you're comfortable accessing a site that is actively telling you no; unauthorized scraping has led to lawsuits and, rarely, criminal charges.
- **APIs.** Some sites *want* you to have their data and offer an API that returns it in a clean format. Always check for one first. `requests` works with APIs too.
- **Timing.** Firing off requests as fast as possible can crash a site or get you blocked. Add pauses between requests (e.g. `time.sleep(1)`). Slower is more reliable.

## Summary

| Tool | Use it for |
|---|---|
| `requests.get(url).text` | downloading a page's HTML |
| `str.find()` + slicing | quick one-off lookups (brittle) |
| `re.search()` / `re.finditer()` | finding text that follows a pattern |
| `BeautifulSoup.find()` / `find_all()` | finding elements by tag and class |
| `pd.DataFrame(...)` / `.to_csv()` | turning results into a real dataset |

**Regex cheat sheet:** `+` one or more · `*` zero or more · `?` zero or one · `.` any character · `^` start · `$` end · `[a-z]` character class · `\d` digit · `\s` whitespace · `\w` word character · `\` escape